# Lab 17: Classification — When 90% Accurate Is Worthless
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 66 min Core + 20 min Write it yourself + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Fit logistic regression and interpret a coefficient as a log-odds and as an odds ratio
- Show accuracy failing as a metric under class imbalance
- Move the decision threshold and trace what it does to precision and recall
- Choose a threshold from the cost of each error rather than from 0.5
- Write a small function that gives a classifier's precision and recall at any threshold

**Dataset:** Simulated loan defaults at low prevalence (measured: a 9.6% default rate, so a model
that approves everyone is 90.4% accurate)

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions.
Part 4 is yours: two one-line blanks, then one question. "Write it yourself" is required: a short
function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**Plan on about 1¾ hours** (1 hour 45 minutes) for the required work: setup, Parts 1-4, Write it
yourself, and the README and submission.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Step 9:** two blanks (`____`), one line each: the expected cost per applicant at a threshold, and the threshold with the lowest cost.
> - **Write it yourself (required),** after Part 4: `precision_recall_at()`, a function that gives back a classifier's precision and recall at any threshold. About three lines from a blank cell. The check cell below it must print "Passed"; it then uses your function at Step 8's closed-form threshold.
> - The answers to the interpretation questions after each part, in text cells: two each for Parts 1 and 2, and one each for Parts 3 and 4.
>
> **You run and read:** everything else. Setup, Parts 1-3 and Step 8 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The CHECK cell after Write it yourself is different: you only run it and read its message; you do not need to follow its code.
>
> **The Extension** (optional) and **the Challenge** (take-home) have no starter code: you write them in new cells, from the task. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Step 2 seeds its own random generator, so running a cell twice gives the same numbers.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the seed
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn import metrics       # accuracy, precision, recall and the confusion matrix

RANDOM_STATE = 42     # the seed for the simulated loans (Step 2) and for the train/test split (Step 3)

print("Setup complete.")

## Part 1: The Model That Learned Nothing (GUIDED — 20 min)

A lender sees 6,000 loan applications, and roughly one in ten will default. Step 2 simulates them,
so the truth is known: each applicant's chance of default comes from a logistic formula in three
features, and Part 2 checks how close a fitted model gets to it.

- `income` is drawn from a lognormal distribution (`rng.lognormal`): always positive, with a long
  right tail, like real incomes.
- `dti`, debt to income, runs from 0 to 0.7: `rng.beta` draws a number between 0 and 1, scaled by 0.7.
- `prior`, the number of earlier defaults, is a count from 0 up (`rng.poisson`).

`logit` is the true log-odds of default. `1 / (1 + np.exp(-logit))`, the logistic function, turns
it into a probability between 0 and 1. `rng.binomial(1, prob)` then flips one coin per applicant,
which lands 1 (default) with that applicant's own probability.

The formula uses the log of income, so the model's feature is `log_income`; the raw `income` column
is kept only for reference.

`pd.DataFrame({...})` builds a table from a dictionary: each name becomes a column. In the print,
`:,` puts commas in the thousands and `.1%` shows a share as a percent with one decimal.
`default.mean()` is the mean of 0s and 1s, so it is the share of 1s: the default rate.

In [ ]:
# GUIDED — run as-is
# Step 2: 6,000 simulated loan applications
rng = np.random.default_rng(seed=RANDOM_STATE)     # seeded here, so rerunning gives the same loans
n = 6_000
income = rng.lognormal(10.8, 0.45, n)
dti = rng.beta(2, 5, n) * 0.7
prior = rng.poisson(0.3, n)
logit = -3.4 - 0.9 * (np.log(income) - 10.8) + 4.0 * dti + 0.6 * prior    # the true log-odds
prob = 1 / (1 + np.exp(-logit))
default = rng.binomial(1, prob)

loans = pd.DataFrame({"income": income, "log_income": np.log(income),
                      "dti": dti, "prior": prior, "default": default})
FEATURES = ["log_income", "dti", "prior"]
print(f"n = {n:,}, default rate = {default.mean():.1%}")

**Expected output:** `n = 6,000, default rate = 9.6%`.

Step 3 splits the applications, as in Lab 12: `train_test_split` keeps 30% of the rows
(`test_size=0.3`) as test rows. `stratify=loans["default"]` deals the defaults out evenly, so the
default rate is the same in both sets. Then it scores the simplest possible rule, one that never
looks at the data: predict "no default" for everyone, and approve every loan.

In [ ]:
# GUIDED — run as-is
# Step 3: keep 30% of the applications for testing, and score the rule that approves everyone
X_tr, X_te, y_tr, y_te = train_test_split(
    loans[FEATURES], loans["default"],
    test_size=0.3, random_state=RANDOM_STATE, stratify=loans["default"])

base_acc = 1 - y_te.mean()       # the accuracy of predicting "no default" for every test applicant
print(f"Accuracy of 'never defaults, approve everyone': {base_acc:.1%}")

Step 4 fits the classifier. `make_pipeline(StandardScaler(), LogisticRegression(...))` is Lab 16's
pattern: the scaler puts each feature on mean 0 and standard deviation 1, learned from the training
rows, and the pipeline scales any rows it is handed later in the same way before the model sees
them. `.fit(X, y)` takes the predictors first, then the outcome (Lab 12). `max_iter=2_000` lets the
fitting routine take more steps to settle. scikit-learn's `LogisticRegression` shrinks its
coefficients a little by default (the chapter's `penalty=None` switches that off), and, because of
the scaler, they are per standard deviation, so not on Part 2's scale; here the model is used only
for its predictions.

There are two ways to ask for a prediction:
- `.predict(X)` gives a 0 or a 1 for each row: 1 when the predicted probability of default is above 0.5.
- `.predict_proba(X)` gives two columns for each row, one probability per class, in the order of the
  classes: column 0 for "no default" (0), column 1 for "default" (1). `[:, 1]` keeps every row (`:`)
  of column 1.

`metrics.confusion_matrix(y_true, y_pred)` takes the true labels first, then the predictions. Its
rows are the truth (0, then 1) and its columns the prediction (0, then 1). `pred.sum()` adds up the
1s: the predicted defaults.

In [ ]:
# GUIDED — run as-is
# Step 4: fit on the training rows, and compare with approving everyone
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=2_000)).fit(X_tr, y_tr)
pred = clf.predict(X_te)                     # 0 or 1 for each test applicant
prob_te = clf.predict_proba(X_te)[:, 1]      # each test applicant's probability of default

print(f"model accuracy    : {metrics.accuracy_score(y_te, pred):.1%}")
print(f"baseline accuracy : {base_acc:.1%}")
print(f"\npredicted defaults: {pred.sum()} of {len(pred)}")
print(metrics.confusion_matrix(y_te, pred))

**Expected output:** `model accuracy : 90.4%`, `baseline accuracy : 90.4%`,
`predicted defaults: 0 of 1800`, and the confusion matrix `[[1628 0] [172 0]]`.

### Interpretation Questions

1. The model's accuracy equals, to the printed decimal, that of a rule that ignores every feature. Explain why accuracy behaves this way at the measured 9.6% prevalence, and say what accuracy that same do-nothing rule would score if prevalence were 1%.
2. Read the confusion matrix. Which error is the model making, and which is it avoiding?

*Your answers here:*

1.
2.

## Part 2: Coefficients on Two Scales (GUIDED — 16 min)

Step 5 fits the same model with statsmodels, which prints the p-values that scikit-learn leaves out,
and applies no penalty. `sm.Logit(y, X)` is the logistic version of Lab 10's `sm.OLS(y, X)`: the
outcome first, and `sm.add_constant` for the intercept. `.fit(disp=False)` fits without printing
progress messages. `.params` and `.pvalues` hold the coefficients and p-values, looked up by name.

A coefficient here is the change in the **log-odds** of default, log(p / (1 − p)), for one more unit
of the feature. `np.exp(coef)` turns it into an **odds ratio**: the number the odds of default are
multiplied by.

In [ ]:
# GUIDED — run as-is
# Step 5: the coefficients as log-odds and as odds ratios
logit_sm = sm.Logit(y_tr, sm.add_constant(X_tr)).fit(disp=False)

# >12 right-aligns text in 12 characters; >10.4f right-aligns a number in 10, with 4 decimals
print(f"{'term':>12}{'coef':>10}{'odds ratio':>13}{'p':>9}")
print("-" * 46)
for name in ["const"] + FEATURES:     # + joins two lists: the constant, then the three features
    b = logit_sm.params[name]
    print(f"{name:>12}{b:>10.4f}{np.exp(b):>13.4f}{logit_sm.pvalues[name]:>9.4f}")

**Expected output:** four rows. `const` 6.2265, `log_income` −0.8707 (odds ratio 0.4187), `dti`
3.2660 (26.2055) and `prior` 0.4861 (1.6260). Every p-value prints as 0.0000. The `log_income`
coefficient is close to the −0.9 in Step 2's formula. The constant's row is not an odds ratio: it is
the baseline odds when every feature is 0, so skip its 505.9923.

An odds ratio is not a change in probability. Step 6 takes the odds ratio for one more prior default
and applies it at three starting probabilities, 2%, 10% and 40%. For each one it turns the
probability into odds, p / (1 − p), multiplies the odds by the odds ratio, and turns the result back
into a probability, odds / (1 + odds). `.0%` prints a share as a percent with no decimals.

In [ ]:
# GUIDED — run as-is
# Step 6: one more prior default, starting from three different probabilities
or_prior = np.exp(logit_sm.params["prior"])     # read from the fit, not typed in
print(f"odds ratio for one more prior default: {or_prior:.2f}")
for p0 in [0.02, 0.10, 0.40]:
    odds = p0 / (1 - p0) * or_prior
    print(f"   from {p0:.0%} -> {odds/(1+odds):.1%}")

**Expected output:** an odds ratio of 1.63, then `from 2% -> 3.2%`, `from 10% -> 15.3%` and
`from 40% -> 52.0%`.

### Interpretation Questions

1. Interpret the `dti` coefficient on both scales, in a sentence each.
2. The same odds ratio produces different probability changes at different baselines. Explain why, and what that means for reporting to a non-technical audience.

*Your answers here:*

1.
2.

## Part 3: The Threshold Is a Decision (GUIDED — 12 min)

`.predict()` declines an applicant only when the predicted probability is above 0.5, and in Step 4
no test applicant's was. Step 7 makes the cutoff a choice. For a threshold `t`, `prob_te >= t` is
True for every applicant whose probability is `t` or more, and `.astype(int)` turns True and False
into 1 and 0, so 1 means "decline". Then it scores those decisions:

- **precision**: of the applicants declined, the share who would have defaulted;
- **recall**: of the applicants who would have defaulted, the share declined.

`metrics.precision_score` and `metrics.recall_score` take the true labels first, then the decisions,
like `confusion_matrix`. When nobody is declined, precision is 0 / 0; `zero_division=0` reports it
as 0 instead of warning. The lists `precisions` and `recalls` keep one entry per threshold, for the
check after "Write it yourself", and `[-1]` reads the entry just added.

In [ ]:
# GUIDED — run as-is
# Step 7: precision, recall and accuracy at six thresholds
THRESHOLDS = [0.5, 0.3, 0.2, 0.12, 0.07, 0.04]
precisions, recalls = [], []
print(f"{'threshold':>11}{'declined':>10}{'precision':>11}{'recall':>9}{'accuracy':>10}")
print("-" * 52)
for t in THRESHOLDS:
    yp = (prob_te >= t).astype(int)          # 1 = decline
    precisions.append(metrics.precision_score(y_te, yp, zero_division=0))
    recalls.append(metrics.recall_score(y_te, yp))
    print(f"{t:>11.2f}{yp.sum():>10}{precisions[-1]:>11.3f}{recalls[-1]:>9.3f}"
          f"{metrics.accuracy_score(y_te, yp):>10.3f}")

**Expected output:** six rows. At 0.50 nobody is declined: precision and recall 0.000, accuracy
0.904. At 0.20, 98 are declined, with precision 0.255, recall 0.145 and accuracy 0.878. At 0.12, 433
are declined (0.180, 0.453, 0.751), and at 0.04, 1669 (0.103, 1.000, 0.168).

### Interpretation Question

1. In Step 4 the model declined nobody, and its accuracy matched a rule that ignores the data. Does this table show that the model learned something anyway? Name the numbers that show it, and say what choosing the threshold with the highest accuracy would do with them.

*Your answer here:*

## Part 4: Price the Errors (YOUR TASK — 18 min)

The two errors cost a lender very different amounts. A false negative, approving a loan that
defaults, loses 60% of a $20,000 principal. A false positive, declining an applicant who would have
repaid, loses the 8% interest margin.

The lecture's closed form gives a threshold from those two costs alone: decline when the probability
of default is at least FP_COST / (FP_COST + FN_COST). Step 8 prints it. In Step 9 you find the
threshold by brute force instead: try every threshold from 0.010 to 0.600, in steps of 0.001, on the
test applicants, and keep the cheapest.

In [ ]:
# GUIDED — run as-is
# Step 8: the cost of each error, and the closed-form threshold
FN_COST = 0.60 * 20_000      # approving a loan that defaults
FP_COST = 0.08 * 20_000      # declining an applicant who would have repaid
print(f"cost of a false negative: {FN_COST:,.0f}")
print(f"cost of a false positive: {FP_COST:,.0f}")
print(f"closed-form threshold   : {FP_COST / (FP_COST + FN_COST):.3f}")

**Expected output:** 12,000 and 1,600, and a closed-form threshold of 0.118.

Step 9 is yours: two blanks, one line each.

`expected_cost(t)` makes Step 7's decisions at threshold `t` and counts the errors. `.ravel()`
flattens the 2 × 2 confusion matrix into its four counts, row by row: true negatives, false
positives, false negatives, true positives. `np.arange(0.01, 0.601, 0.001)` makes the thresholds
0.010, 0.011, ..., 0.600 (the stop value, 0.601, is left out, as in `range`). `np.array(costs)`
turns the list into an array, so that `.min()` works on it. In the last print, `:+.0%` shows a share
as a percent with its sign.

1. In `expected_cost`, give back the expected cost per applicant: what the false negatives and the
   false positives cost in all, spread over every test applicant.
2. `best_t` is the threshold in `grid` with the lowest cost. `np.argmin` gives a position, not a
   threshold.

The cell then prices the library's threshold, 0.5, with your `expected_cost`.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 9: the expected cost at every threshold from 0.010 to 0.600, and the cheapest one
def expected_cost(t):
    """Expected cost per test applicant when everyone with probability t or more is declined."""
    yp = (prob_te >= t).astype(int)
    tn, fp, fn, tp = metrics.confusion_matrix(y_te, yp).ravel()
    return ____                      # blank 1

grid = np.arange(0.01, 0.601, 0.001)
costs = []
for t in grid:
    costs.append(expected_cost(t))
costs = np.array(costs)

best_t = ____                        # blank 2: the threshold in grid with the lowest cost
best_cost = costs.min()
cost_at_half = expected_cost(0.5)    # the library's threshold, for comparison

print(f"cost-minimising threshold  : {best_t:.3f}")
print(f"expected cost per applicant: {best_cost:,.0f}")
print(f"cost at the default 0.5    : {cost_at_half:,.0f}")
print(f"the default costs           {cost_at_half / best_cost - 1:+.0%} more per applicant")

**Expected output:** a cost-minimising threshold of 0.103, an expected cost of 926 per applicant,
1,147 at 0.5, and `+24% more per applicant`. Near-misses: a threshold of 93.000 is its position in
`grid`, not the threshold. Costs in the millions are totals over all 1,800 test applicants, not
costs per applicant. 278 per applicant means you divided by all 6,000 loans, not the 1,800 test
rows. A threshold of 0.455 with costs near 150 means the two costs are swapped; a cost of 0 means
you counted errors, not dollars.

### Interpretation Question

1. How far is your cost-minimising threshold from 0.5, and from Step 8's closed form? Then say what happens to the threshold if the recovery rate on a default improves from 40% to 70%, and why that direction makes sense for a lender.

*Your answer here:*

---

## Write it yourself (required — 20 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** Step 7 scored one model, on the test rows, from probabilities already stored in
`prob_te`. Turn it into a function you can point at any model, any rows and any threshold.

Write a function `precision_recall_at(model, X, y, t)` that declines every row of `X` whose
predicted probability of default, from `model`, is `t` or more, and gives back two numbers: the
precision, then the recall, of those decisions against the true labels `y`. With Step 4's model and
the test rows, `precision_recall_at(clf, X_te, y_te, 0.2)` gives Step 7's 0.255 and 0.145.

- It takes about three lines: the `def` line, a line that makes the 0/1 decisions, and the `return`.
- Step 7 has the pieces. Inside the function, use the four arguments wherever Step 7 uses its own
  probabilities, labels and threshold.
- The probabilities come from `model` and `X`, not from `prob_te`. Step 4 shows which column of
  `predict_proba` holds the probability of default.
- `return a, b` gives back two values together. Give back the full numbers; round only when you print.
- If you compute a share by hand, Python may display it as `np.float64(0.2551...)`. That is just a
  number.

Then run the check cell. It calls your function at Step 7's thresholds and compares the answers with
Step 7's table; it also tries it on other rows, another model and other thresholds. `assert` stops
with its message when its condition is False; the message tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): precision_recall_at(model, X, y, t)

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "precision_recall_at" not in globals():
    print("Not written yet: write precision_recall_at() in the cell above and run it, then run this check.")
else:
    import copy, re

    # Step 4's clf must still give Step 4's probabilities; a function that refits it changes them
    assert np.allclose(clf.predict_proba(X_te)[:, 1], prob_te), (
        "Step 4's clf has been refitted, perhaps by an earlier version of your function. Only use "
        "predict_proba in the function. Rerun Step 4, then this cell.")

    def call(model, X, y, t):
        """Run your precision_recall_at on a copy of the model; if it goes wrong, say where to look."""
        m = copy.deepcopy(model)                  # a copy, so your function cannot change clf
        before = m.predict_proba(X)
        # try/except catches an error inside your function, so the message can name the cause
        try:
            out = precision_recall_at(m, X, y, t)
        except Exception as e:
            hint = "Read the error above: it says what went wrong."
            sizes = [int(s) for s in re.findall(r"\d+", str(e))]   # the row counts in the message
            if "positional argument" in str(e) or "predict_proba" in str(e):
                hint = "precision_recall_at takes four arguments, in this order: model, X, y, t."
            elif "inconsistent numbers of samples" in str(e) and sizes and min(sizes) <= 2:
                hint = "[1] or [:1] picks rows of predict_proba; you want a column, as in Step 4."
            elif "inconsistent numbers of samples" in str(e):
                hint = (f"It was handed {len(X):,} rows and mixed them with a different number. Does it "
                        "use its own X and y everywhere, or Step 7's prob_te and y_te?")
            elif "multilabel" in str(e) or "multioutput" in str(e):
                hint = ("Its decisions have two columns. predict_proba gives one column per class: "
                        "use only the probability of default.")
            # str(e)[:200] keeps the first 200 characters of the error message
            raise AssertionError(f"precision_recall_at on {len(X):,} rows at t = {t} stopped with an "
                                 f"error:\n  {str(e)[:200]}\n{hint}") from None
        assert np.allclose(m.predict_proba(X), before), (
            "Your function refits the model; only use predict_proba. Rerun Step 4, then this cell.")
        assert np.allclose(clf.predict_proba(X_te)[:, 1], prob_te), (
            "Your function refits Step 4's clf; only use predict_proba. Rerun Step 4, then this cell.")
        return out

    def two_numbers(out, rows):
        """Check that out is two numbers, and hand them back as plain floats."""
        assert out is not None, "precision_recall_at gave back nothing. Does it end with a return line?"
        assert not hasattr(out, "predict_proba"), (
            "precision_recall_at gives back the model. Give back the precision and the recall.")
        assert np.size(out) != rows, (
            "precision_recall_at gives back one value per row: the decisions. Score them, and give "
            "back the precision and the recall.")
        assert np.size(out) == 2, (f"precision_recall_at gives back {np.size(out)} value(s). Give back "
                                   "two numbers: the precision, then the recall.")
        p, r = np.ravel(np.asarray(out, dtype=float))
        return p, r

    k = THRESHOLDS.index(0.2)              # where Step 7 stored threshold 0.2
    P, R = precisions[k], recalls[k]       # Step 7's 0.255 and 0.145
    p, r = two_numbers(call(clf, X_te, y_te, 0.2), len(X_te))

    # np.isnan is True for "not a number", which is what 0 / 0 gives
    assert not np.isnan(p), (
        "Your precision is 0 / 0: nobody was declined at t = 0.2. .predict() always cuts at 0.5; "
        "make the decisions from the probabilities and t.")
    assert not (np.isclose(p, 100 * P) or np.isclose(r, 100 * R)), (
        "Your numbers are percents. Give back shares between 0 and 1, as Step 7 does.")
    assert p <= 1 and r <= 1, (
        f"precision_recall_at gives back {p:g} and {r:g}: counts, not shares. Precision and recall "
        "are shares between 0 and 1; divide each count by the right total.")
    assert not (np.isclose(p, R) and np.isclose(r, P)), (
        "Your precision is Step 7's recall, and your recall its precision. Check the order in which "
        "you pass the true labels and the decisions, and the order in which you give back the two numbers.")
    assert not np.isclose(r, 1), (
        "At t = 0.2 your function declines every applicant who defaulted, far more than Step 7 "
        "did. Is it reading the probability of NO default? Step 4 shows which column of "
        "predict_proba holds the probability of default.")

    # nobody's probability of default reaches 0.5, so 0.2 and 0.07 should give different answers
    same_t = np.allclose(two_numbers(call(clf, X_te, y_te, 0.07), len(X_te)), (p, r))
    assert not (same_t and p == 0 and r == 0), (
        "Nobody is declined, at any threshold: .predict() always cuts at 0.5. Make the decisions "
        "from the probabilities and t.")
    assert not same_t, (
        "precision_recall_at gives the same answer at 0.2 and at 0.07. Does it use its t argument?")

    # the 20th-highest probability: at exactly that threshold, its applicant should be declined
    t_tie = float(np.sort(prob_te)[-20])
    at_tie = two_numbers(call(clf, X_te, y_te, t_tie), len(X_te))
    just_below = two_numbers(call(clf, X_te, y_te, np.nextafter(t_tie, 0)), len(X_te))
    assert np.allclose(at_tie, just_below), (
        "At a threshold equal to an applicant's own probability, your function approves that "
        "applicant. The task declines every row whose probability is t or more.")

    assert not np.allclose(two_numbers(call(clf, X_tr, y_tr, 0.2), len(X_tr)), (p, r)), (
        "precision_recall_at gives the same answer on the training rows as on the test rows. Does it "
        "use its X and y arguments, or Step 7's prob_te and y_te?")
    # a second model: the same features, with the defaults weighted up, so its probabilities run higher
    other = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000, class_weight="balanced"))
    other.fit(X_tr, y_tr)
    assert not np.allclose(two_numbers(call(other, X_te, y_te, 0.2), len(X_te)), (p, r)), (
        "precision_recall_at gives the same answer for a different model. Does it use its model "
        "argument, or Step 4's clf?")

    for i, t in enumerate(THRESHOLDS):
        if t == 0.5:
            continue                       # nobody is declined at 0.5, so precision is 0 / 0 there
        mine = two_numbers(call(clf, X_te, y_te, t), len(X_te))
        # round(v, places) keeps that many decimals; try 2 to 5 of them
        for places in [2, 3, 4, 5]:
            rounded = (round(precisions[i], places), round(recalls[i], places))
            assert not (np.allclose(mine, rounded, rtol=0, atol=1e-12)
                        and not np.allclose(mine, (precisions[i], recalls[i]), rtol=1e-9, atol=0)), (
                "Your numbers are rounded. Give back the full numbers; round only when you print.")
        assert np.allclose(mine, (precisions[i], recalls[i]), rtol=1e-9, atol=0), (
            f"precision_recall_at(clf, X_te, y_te, {t}) gives {mine[0]:.3f} and {mine[1]:.3f}; Step 7 "
            f"printed {precisions[i]:.3f} and {recalls[i]:.3f}. Check three things against Step 7: "
            "the probabilities, the comparison with t, and which numbers each share divides.")

    t_closed = FP_COST / (FP_COST + FN_COST)
    p_c, r_c = two_numbers(call(clf, X_te, y_te, t_closed), len(X_te))
    print("Passed. precision_recall_at matches Step 7 at every threshold, and uses the model, rows "
          "and threshold it is given.")
    print(f"At Step 8's closed-form threshold, {t_closed:.3f}: precision {p_c:.3f}, recall {r_c:.3f}")

---

## Extension (Optional — 15 min): Calibration

There is no starter code: write it in the cell below.

Are the probabilities honest? Among applicants given about a 10% chance of default, about 10% should
default. Sort the test applicants into ten bins by predicted probability, and in each bin compare
the mean prediction with the share who actually defaulted: a reliability diagram.
`calibration_curve(y_te, prob_te, n_bins=10, strategy="quantile")`, from `sklearn.calibration`,
gives back both, observed share first; `strategy="quantile"` puts the same number of applicants in
each bin. Plot the observed share against the mean prediction, with the 45-degree line for reference
(`import matplotlib.pyplot as plt`).

**Your answer.** Are the probabilities calibrated? Say what you would do if they were not.

In [ ]:
# EXTENSION (optional) — write your reliability diagram here

*Your answer here:*

---

## Challenge (Take-Home): Resampling the Minority Class

There is no starter code: write your answer in the cell below and in new cells under it.

1. Refit Step 4's pipeline with `LogisticRegression(class_weight="balanced", max_iter=2_000)`, which
   weights the defaults so that, as a group, they count as much as the repayments.
2. Compare precision, recall and the cost-minimising threshold with the unweighted model.
3. The balanced model's probabilities are no longer calibrated to the true prevalence. Show it.
4. Argue whether balancing was worth it, given that you can move the threshold instead.

If you skip the Challenge, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Worked with 6,000 simulated loan applications with a default rate
  of [YOUR VALUE], where a rule that approves everyone is
  [YOUR VALUE] accurate
* Fitted a logistic regression; at the usual 0.5 threshold its
  accuracy was [YOUR VALUE], with [YOUR VALUE] predicted defaults
* Read the coefficients as log-odds and as odds ratios, and showed
  that one odds ratio ([YOUR VALUE] for one more prior default)
  means different changes in probability at different starting points
* Traced precision and recall across six thresholds
* Chose a threshold by minimising the expected cost of the two errors
* Wrote a small function, precision_recall_at(), that gives a
  classifier's precision and recall at any threshold
* Key findings: cost-minimising threshold [YOUR VALUE], against a
  closed-form [YOUR VALUE]; expected cost [YOUR VALUE] per applicant,
  against [YOUR VALUE] at 0.5; at the closed-form threshold, precision
  [YOUR VALUE] and recall [YOUR VALUE]
* (Only if I did the Extension) Calibration: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Classification: When 90% Accurate Is Worthless
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab17-classification`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab17-classification`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab17-classification`, branch `main`,
   commit message `Lab 17 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.